# Hybrid Low-Resource TTS Dataset Synthesis Pipeline

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Runs the format conversion, denoising, trimming, normalization, WPM filtering, voice-cloning and punctuation-restoration stages that turn a raw ASR-style donor corpus into a voice-converted TTS dataset. Denoising uses [audiosronnx](https://github.com/TigreGotico/audiosronnx) and voice conversion uses [voiceclonnx](https://github.com/TigreGotico/voiceclonnx); both run on `onnxruntime` alone, so no PyTorch or CUDA driver is needed for either stage.

| Field | Value |
|---|---|
| Inputs | a directory of raw donor audio (`RAW_AUDIO_SOURCE`), its transcript (`METADATA_CSV`) and one reference WAV per target voice (`REF_VOICES_DIR`) |
| Outputs | a voice-converted LJSpeech-style dataset and a `manifest.jsonl` ([manifest.md](../docs/manifest.md)) under `WORK_DIR` |
| Runtime | full run: minutes to tens of minutes depending on dataset size and the chosen `DENOISE_ENGINE`/`VC_ENGINE`, CPU-only via `onnxruntime`; CPU smoke path (`SMOKE=True`, `N_SAMPLES` files): a few minutes end to end, including denoising and voice conversion, with punctuation restoration skipped |
| Hardware | CPU (`onnxruntime`); `ONNX_RUNTIME=cuda` or `rocm` speeds up denoising and voice conversion on their own hardware, but neither needs one |
| Cost | CPU, under 10 minutes on the smoke path |

CI: executed on the smoke path

**Read more:**

- [OVOS TTS plugins](https://openvoiceos.github.io/ovos-technical-manual/320-tts_plugins/)
- [phoonnx](https://github.com/TigreGotico/phoonnx)
- [voiceclonnx](https://github.com/TigreGotico/voiceclonnx)
- [audiosronnx](https://github.com/TigreGotico/audiosronnx)
- [Making synthetic voices from scratch](https://openvoiceos.github.io/ovos-blogs/posts/2025-06-26-making-synthetic-voices-from-scratch)


In [ ]:
SMOKE = False
N_SAMPLES = 32
# EPOCHS does not apply: no training happens in this notebook.


In [ ]:
import os


def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return os.path.abspath(f"./{name}")


def _env(key, default=""):
    return os.environ.get(key, default)


def _parse_typed_env(env_name, raw, kind):
    if kind == "bool":
        low = raw.strip().lower()
        if low in ("1", "true", "yes", "on"):
            return True
        if low in ("0", "false", "no", "off"):
            return False
        raise ValueError(f"{env_name}={raw!r} is not a valid bool (use true/false)")
    if kind == "int":
        try:
            return int(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid int") from None
    if kind == "float":
        try:
            return float(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid float") from None
    return raw


def _resolve_onnx_providers(raw):
    """A comma list, or "auto": CUDA, ROCm, MIGraphX, CPU, CPU always last."""
    raw = (raw or "").strip()
    if not raw or raw.lower() == "auto":
        import onnxruntime as ort

        preferred = [
            "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider",
            "CPUExecutionProvider",
        ]
        available = ort.get_available_providers()
        resolved = [p for p in preferred if p in available]
        if "CPUExecutionProvider" not in resolved:
            resolved.append("CPUExecutionProvider")
        return resolved
    return [p.strip() for p in raw.split(",") if p.strip()]


# ==========================================
# GLOBAL CONFIGURATION (ENV VARS)
# ==========================================

os.environ.setdefault("WORK_DIR", _default_base_dir("asr2tts_workspace"))
os.makedirs(os.environ["WORK_DIR"], exist_ok=True)

# Input data sources. No default names a real dataset: SMOKE=True builds a
# tiny fixture set below, and SMOKE=False needs every one of these set to a
# real donor corpus.
os.environ.setdefault("RAW_AUDIO_SOURCE", "")
os.environ.setdefault("METADATA_CSV", "")
os.environ.setdefault("REF_VOICES_DIR", "")

# The BCP-47 tag written into manifest.jsonl's "lang" field (docs/manifest.md).
os.environ.setdefault("TTS_LANG", "en-US")

# Audio denoising (Step 2, audiosronnx load_denoise). audiosronnx.available_denoisers()
# prints every alias; the default, gtcrn, is a 23.7K-parameter, ~0.5 MB model chosen
# so the smoke path never downloads a large file.
os.environ.setdefault("DENOISE_ENGINE", "gtcrn")

# Voice conversion (Step 6, voiceclonnx). `voiceclonnx list` prints every alias; the
# default, openvoice, measures 0% WER quantized or not, and is the smallest quantized
# download among the 0%-WER engines.
os.environ.setdefault("VC_ENGINE", "openvoice")
os.environ.setdefault("VC_QUANTIZED", "true")

# onnxruntime, shared by audiosronnx and voiceclonnx.
os.environ.setdefault("ONNX_RUNTIME", "cpu")     # cpu|cuda|rocm - which onnxruntime build the install cells fetch
os.environ.setdefault("ONNX_PROVIDERS", "auto")  # execution providers: a comma list, or "auto"

# Punctuation restoration (Step 7).
os.environ.setdefault("PUNCT_MODEL", "kredor/punctuate-all")

# Processing parameters (Steps 3-5)
os.environ.setdefault("TARGET_LUFS", "-20.0")
os.environ.setdefault("WPM_STD_THRESHOLD", "2.0")
os.environ.setdefault("MIN_TRIM_DURATION", "1.0")

if SMOKE:
    # Generate a handful of short sine-tone WAV fixtures so the smoke path has
    # something to convert, denoise, trim and voice-convert, without a real
    # donor dataset or reference voices.
    import array
    import math
    import wave

    sample_rate = 16000

    def _write_tone(path, freq, seconds):
        n = sample_rate * seconds
        samples = array.array("h", [
            int(3000 * math.sin(2 * math.pi * freq * t / sample_rate)) for t in range(n)
        ])
        with wave.open(path, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.writeframes(samples.tobytes())

    smoke_src = os.path.join(os.environ["WORK_DIR"], "smoke_raw_audio")
    os.makedirs(smoke_src, exist_ok=True)
    for i in range(N_SAMPLES):
        _write_tone(os.path.join(smoke_src, f"sample_{i:02d}.wav"), 220, 2)
    os.environ["RAW_AUDIO_SOURCE"] = smoke_src

    smoke_csv = os.path.join(os.environ["WORK_DIR"], "smoke_metadata.csv")
    with open(smoke_csv, "w") as f:
        for i in range(N_SAMPLES):
            f.write(f"sample_{i:02d}|this is smoke fixture sentence {i}\n")
    os.environ["METADATA_CSV"] = smoke_csv

    smoke_refs = os.path.join(os.environ["WORK_DIR"], "smoke_ref_voices")
    os.makedirs(smoke_refs, exist_ok=True)
    for name, freq in (("ref_a", 180), ("ref_b", 260)):
        _write_tone(os.path.join(smoke_refs, f"{name}.wav"), freq, 1)
    os.environ["REF_VOICES_DIR"] = smoke_refs
elif not (os.environ["RAW_AUDIO_SOURCE"] and os.environ["METADATA_CSV"] and os.environ["REF_VOICES_DIR"]):
    raise RuntimeError(
        "RAW_AUDIO_SOURCE, METADATA_CSV and REF_VOICES_DIR must each be set to a real "
        "path when SMOKE=False; there is no placeholder default."
    )

# Define pipeline subdirectories (chain of custody)
base = os.environ["WORK_DIR"]
os.environ["DIR_CONVERTED"] = os.path.join(base, "1_converted")
os.environ["DIR_DENOISED"] = os.path.join(base, "2_denoised")
os.environ["DIR_TRIMMED"] = os.path.join(base, "3_trimmed")
os.environ["DIR_NORMALIZED"] = os.path.join(base, "4_normalized")
os.environ["DIR_FILTERED"] = os.path.join(base, "5_wpm_filtered")
os.environ["DIR_FINAL_VC"] = os.path.join(base, "6_vc_output")

print(f"Workspace: {os.environ['WORK_DIR']}")


---
### Step 1: Format Standardization
Converts mixed audio formats (mp3, ogg, flv, etc.) to standard WAV format.


In [ ]:
%pip install --quiet --pre "pydub>=0.25.1"

import os
import shutil
from pydub import AudioSegment

INPUT_DIR = _env("RAW_AUDIO_SOURCE")
OUTPUT_DIR = _env("DIR_CONVERTED")

SUPPORTED_FORMATS = {'.mp3', '.mp4', '.wma', '.ogg', '.flv', '.aac', '.m4a', '.amr', '.wav'}

def convert_dataset(input_path, output_path):
    os.makedirs(output_path, exist_ok=True)
    print(f"Converting from {input_path} -> {output_path}")

    for root, _, files in os.walk(input_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            base, ext = os.path.splitext(filename)

            if ext.lower() not in SUPPORTED_FORMATS:
                continue

            target_file = os.path.join(output_path, f"{base}.wav")

            if ext.lower() == '.wav':
                shutil.copy2(file_path, target_file)
            else:
                audio = AudioSegment.from_file(file_path)
                audio.export(target_file, format='wav')
                print(f"Converted: {filename}")

convert_dataset(INPUT_DIR, OUTPUT_DIR)


### Step 2: Audio Denoising

Removes background noise with [audiosronnx](https://github.com/TigreGotico/audiosronnx)'s
`load_denoise`, a pure-`onnxruntime` denoiser: no PyTorch or CUDA driver is needed at
runtime. `DENOISE_ENGINE` names the engine (`audiosronnx.available_denoisers()` prints the
rest); `DENOISE_<OPTION>` sets one constructor option of the selected engine, and setting an
option the engine does not accept stops the notebook, naming the engine and the option.


In [ ]:
%pip install --quiet --pre "audiosronnx>=0.9.0a2" "tqdm>=4.70.1"

onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()
if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"

import inspect
import os

from audiosronnx import load_denoise
from audiosronnx.base import get_engine
from tqdm import tqdm

# option name -> typed-parsing kind; the env var is DENOISE_<OPTION UPPERCASE>
DENOISE_OPTION_TYPES = {
    "revision": "str",
    "onnx_path": "str",
    "mask_only": "bool",
    "enc_path": "str",
    "erb_dec_path": "str",
    "df_dec_path": "str",
    "model": "str",
    "attn_limit_db": "float",
    "dither": "float",
    "analysis_path": "str",
    "vocoder_path": "str",
    "mel_filters_path": "str",
}

INPUT_DIR = _env("DIR_CONVERTED")
OUTPUT_DIR = _env("DIR_DENOISED")
DENOISE_ENGINE = _env("DENOISE_ENGINE", "gtcrn")
os.makedirs(OUTPUT_DIR, exist_ok=True)

engine_entry = get_engine(DENOISE_ENGINE)
adapter_params = inspect.signature(engine_entry.adapter_class.__init__).parameters
denoise_kwargs = {}
for option, kind in DENOISE_OPTION_TYPES.items():
    env_name = f"DENOISE_{option.upper()}"
    raw = _env(env_name)
    if not raw:
        continue
    if option not in adapter_params:
        raise RuntimeError(f"{env_name} is set but engine {DENOISE_ENGINE!r} has no {option!r} option")
    denoise_kwargs[option] = _parse_typed_env(env_name, raw, kind)
if "providers" in adapter_params:
    denoise_kwargs["providers"] = _resolve_onnx_providers(_env("ONNX_PROVIDERS"))

print(f"Loading audiosronnx denoiser {DENOISE_ENGINE!r}...")
denoiser = load_denoise(DENOISE_ENGINE, **denoise_kwargs)

files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".wav"))
for fname in tqdm(files, desc="Denoising Audio"):
    out_path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(out_path):
        continue
    denoiser.denoise_file(os.path.join(INPUT_DIR, fname), out_path)

print(f"Denoising: {len(files)} file(s) -> {OUTPUT_DIR}")


### Step 3: Silence Trimming
Trims leading/trailing silence using `librosa`. Removes files that are too short after trimming.


In [ ]:
%pip install --quiet --pre "librosa>=1.0.0" "soundfile>=0.14.0"

import os
import librosa
import soundfile as sf
from glob import glob
from tqdm import tqdm

INPUT_DIR = _env("DIR_DENOISED")
OUTPUT_DIR = _env("DIR_TRIMMED")
MIN_DURATION = float(_env("MIN_TRIM_DURATION", "1.0"))

def trim_silence(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = glob(os.path.join(input_dir, '*.wav'))

    for fpath in tqdm(files, desc="Trimming Silence"):
        fname = os.path.basename(fpath)
        out_path = os.path.join(output_dir, fname)

        audio, sr = librosa.load(fpath, sr=None, mono=True)
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=40)
        duration = len(trimmed_audio) / sr

        if duration >= MIN_DURATION:
            sf.write(out_path, trimmed_audio, sr)

trim_silence(INPUT_DIR, OUTPUT_DIR)


### Step 4: Volume Normalization
Normalizes audio to a target loudness (e.g., -20.0 LUFS/dBFS) to ensure consistency.


In [ ]:
%pip install --quiet --pre "pydub>=0.25.1"

import os
from pydub import AudioSegment
from glob import glob
from tqdm import tqdm

INPUT_DIR = _env("DIR_TRIMMED")
OUTPUT_DIR = _env("DIR_NORMALIZED")
TARGET_DBFS = float(_env("TARGET_LUFS", "-20.0"))

def normalize_volume(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = glob(os.path.join(input_dir, '*.wav'))

    for fpath in tqdm(files, desc="Normalizing Volume"):
        fname = os.path.basename(fpath)
        out_path = os.path.join(output_dir, fname)

        audio = AudioSegment.from_wav(fpath)
        gain = TARGET_DBFS - audio.dBFS
        normalized = audio.apply_gain(gain)
        normalized.export(out_path, format="wav")

normalize_volume(INPUT_DIR, OUTPUT_DIR)


### Step 5: WPM (Words Per Minute) Filtering
Filters out outliers based on speaking rate using the provided transcript CSV.


In [ ]:
%pip install --quiet --pre "pandas>=2.2" "soundfile>=0.14.0"

import os
import re
import shutil
import pandas as pd
import soundfile as sf

INPUT_DIR = _env("DIR_NORMALIZED")
CSV_PATH = _env("METADATA_CSV")
OUTPUT_DIR = _env("DIR_FILTERED")
STD_THRESHOLD = float(_env("WPM_STD_THRESHOLD", "2.0"))

def calculate_wpm(row, audio_dir):
    file_path = os.path.join(audio_dir, str(row['file_name']))
    if not file_path.endswith('.wav') and not os.path.exists(file_path):
        file_path += ".wav"

    cleaned_transcript = re.sub(r'[^\w\s]', '', str(row['transcript'])).strip().lower()
    word_count = len(cleaned_transcript.split())

    if word_count == 0 or not os.path.exists(file_path):
        return 0.0

    try:
        with sf.SoundFile(file_path) as f:
            duration_min = (len(f) / f.samplerate) / 60.0
        return word_count / duration_min if duration_min > 0 else 0.0
    except Exception:
        return 0.0

def filter_wpm(input_dir, csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(csv_path, sep='|', names=['file_name', 'transcript'], header=None)

    print("Calculating WPM...")
    df['WPM'] = df.apply(lambda row: calculate_wpm(row, input_dir), axis=1)
    valid_data = df[df['WPM'] > 0]

    mean_wpm = valid_data['WPM'].mean()
    std_wpm = valid_data['WPM'].std()
    lower = mean_wpm - (STD_THRESHOLD * std_wpm)
    upper = mean_wpm + (STD_THRESHOLD * std_wpm)

    print(f"Mean WPM: {mean_wpm:.2f} | Range: {lower:.2f} - {upper:.2f}")

    filtered = valid_data[(valid_data['WPM'] >= lower) & (valid_data['WPM'] <= upper)]

    print(f"Copying {len(filtered)} valid files...")
    new_meta = []
    for _, row in filtered.iterrows():
        fname = str(row['file_name'])
        if not fname.endswith('.wav'):
            fname += ".wav"

        src = os.path.join(input_dir, fname)
        dst = os.path.join(output_dir, fname)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            new_meta.append(f"{fname}|{row['transcript']}")

    with open(os.path.join(output_dir, "metadata_filtered.csv"), "w") as f:
        f.write("\n".join(new_meta))

filter_wpm(INPUT_DIR, CSV_PATH, OUTPUT_DIR)


### Step 6: Voice Conversion

Converts every filtered clip to one or more target voices with
[voiceclonnx](https://github.com/TigreGotico/voiceclonnx), a pure-`onnxruntime`
voice-conversion library: no PyTorch or CUDA driver is needed at runtime. Every WAV
in `REF_VOICES_DIR` names one target voice; each filtered clip is converted to every
reference voice and written under its own subdirectory of `DIR_FINAL_VC`.

`VC_ENGINE` names the engine (`voiceclonnx list` prints the rest); `VC_<OPTION>` sets
one constructor option of the selected engine (`VC_QUANTIZED`, `VC_EXAGGERATION`,
`VC_ODE_STEPS`, `VC_CHUNK_SAMPLES`, `VC_K`, `VC_DEFAULT_MODEL`, `VC_SAMPLE_RATE`,
`VC_SPEAKER_ID`, `VC_F0_UP_KEY`), and setting one the engine does not accept stops
the notebook, naming the engine and the option.


In [ ]:
%pip install --quiet --pre "voiceclonnx>=0.0.3a3"

onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()
if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"

import inspect
import os

from voiceclonnx import VoiceCloner
from voiceclonnx.engines.base import get_engine

# option name -> typed-parsing kind; the env var is VC_<OPTION UPPERCASE>
VC_OPTION_TYPES = {
    "quantized": "bool",
    "exaggeration": "float",
    "ode_steps": "int",
    "chunk_samples": "int",
    "k": "int",
    "default_model": "str",
    "sample_rate": "int",
    "speaker_id": "int",
    "f0_up_key": "int",
}

INPUT_DIR = _env("DIR_FILTERED")
OUTPUT_BASE = _env("DIR_FINAL_VC")
REFS_FOLDER = _env("REF_VOICES_DIR")
VC_ENGINE = _env("VC_ENGINE", "openvoice")

ref_voices = sorted(f for f in os.listdir(REFS_FOLDER) if f.endswith(".wav"))
if not ref_voices:
    raise RuntimeError(f"REF_VOICES_DIR={REFS_FOLDER} has no .wav reference voices")

source_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".wav"))
if not source_files:
    raise RuntimeError(f"{INPUT_DIR} has no filtered .wav clips to convert")

if SMOKE:
    # bound the smoke path regardless of how many voices REF_VOICES_DIR holds,
    # and keep its download small even if VC_QUANTIZED was overridden.
    ref_voices = ref_voices[:2]
    os.environ["VC_QUANTIZED"] = "true"

engine_entry = get_engine(VC_ENGINE)
adapter_params = inspect.signature(engine_entry.adapter_class.__init__).parameters
vc_kwargs = {}
for option, kind in VC_OPTION_TYPES.items():
    env_name = f"VC_{option.upper()}"
    raw = _env(env_name)
    if not raw:
        continue
    if option not in adapter_params:
        raise RuntimeError(f"{env_name} is set but engine {VC_ENGINE!r} has no {option!r} option")
    vc_kwargs[option] = _parse_typed_env(env_name, raw, kind)
if "providers" in adapter_params:
    vc_kwargs["providers"] = _resolve_onnx_providers(_env("ONNX_PROVIDERS"))

print(f"Loading voiceclonnx engine {VC_ENGINE!r}...")
cloner = VoiceCloner(engine=VC_ENGINE, **vc_kwargs)

converted = 0
for ref_voice in ref_voices:
    voice_id = os.path.splitext(ref_voice)[0]
    ref_path = os.path.join(REFS_FOLDER, ref_voice)
    voice_out_dir = os.path.join(OUTPUT_BASE, voice_id)
    os.makedirs(voice_out_dir, exist_ok=True)

    for fname in source_files:
        out_file = os.path.join(voice_out_dir, fname)
        if os.path.exists(out_file):
            continue
        cloner.clone_voice(os.path.join(INPUT_DIR, fname), ref_path, out_file)
        converted += 1

print(f"Voice conversion: {converted} file(s) written across {len(ref_voices)} voice(s) -> {OUTPUT_BASE}")


---
### Step 7: Punctuation Restoration

Restores punctuation on the transcript that every `DIR_FINAL_VC` voice shares
(`DIR_FILTERED`'s `metadata_filtered.csv`; voice conversion changes only the audio,
never the text) with [deepmultilingualpunctuation](https://github.com/oliverguhr/deepmultilingualpunctuation).
No in-house package restores punctuation, so this step keeps that third-party
transformers model; `PUNCT_MODEL` names the Hugging Face token-classification
checkpoint. The result is written to `metadata_punct.csv` under `WORK_DIR`.

Long or unpunctuated transcripts benefit most, since a later training step may
split on sentence boundaries.

Skipped on the smoke path: the default model downloads over 1 GB of weights.


In [ ]:
import os

if SMOKE:
    print(f"Punctuation restoration skipped on the smoke path: {_env('PUNCT_MODEL')} "
          f"downloads over 1 GB of weights.")
else:
    %pip install --quiet --pre "deepmultilingualpunctuation>=1.0.1" "tf-keras>=2.21.0" "transformers>=4.30"

    import pandas as pd
    from deepmultilingualpunctuation import PunctuationModel
    from transformers import pipeline as _hf_pipeline

    CSV_PATH = os.path.join(_env("DIR_FILTERED"), "metadata_filtered.csv")
    OUTPUT_PATH = os.path.join(_env("WORK_DIR"), "metadata_punct.csv")
    PUNCT_MODEL = _env("PUNCT_MODEL", "kredor/punctuate-all")

    def _load_punctuation_model(model_id):
        # deepmultilingualpunctuation's __init__ calls pipeline(..., grouped_entities=False),
        # an argument current transformers dropped for aggregation_strategy; build the same
        # pipeline directly and attach it the way __init__ would.
        model = PunctuationModel.__new__(PunctuationModel)
        model.pipe = _hf_pipeline("ner", model=model_id, aggregation_strategy="none")
        return model

    df = pd.read_csv(CSV_PATH, sep='|', names=['file_name', 'transcript'], header=None)
    model = _load_punctuation_model(PUNCT_MODEL)

    def restore_punct(text):
        if pd.isna(text) or str(text).strip() == "":
            return text
        try:
            return model.restore_punctuation(str(text))
        except Exception:
            return text

    print("Restoring punctuation...")
    df['transcript'] = df['transcript'].apply(restore_punct)

    df.to_csv(OUTPUT_PATH, sep='|', index=False, header=False)
    print(f"Done! Saved to {OUTPUT_PATH}")


---
### Step 8: Manifest

Writes `manifest.jsonl` and its `.meta.json` sidecar under `WORK_DIR`, following
[manifest.md](../docs/manifest.md): one row per voice-converted clip, with `id`,
`text`, `lang`, `audio`, `duration_s`, `sample_rate`, `source` and `speaker` (the
voice identity from `REF_VOICES_DIR`). `text` comes from `metadata_punct.csv` when
Step 7 ran, otherwise from the pre-punctuation `metadata_filtered.csv`. Each
`DIR_FINAL_VC` voice directory also gets its own LJSpeech-style `metadata.csv`,
for tools that expect that format directly.


In [ ]:
import csv
import json
import os
from importlib.metadata import version as _pkg_version

import soundfile as sf

punct_csv = os.path.join(_env("WORK_DIR"), "metadata_punct.csv")
filtered_csv = os.path.join(_env("DIR_FILTERED"), "metadata_filtered.csv")
text_csv = punct_csv if os.path.isfile(punct_csv) else filtered_csv

transcripts = {}
with open(text_csv, newline="", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="|"):
        if len(row) < 2:
            continue
        fname = row[0] if row[0].endswith(".wav") else row[0] + ".wav"
        transcripts[fname] = row[1]

corpus_name = os.path.basename(os.path.normpath(_env("RAW_AUDIO_SOURCE")))
lang = _env("TTS_LANG", "en-US")
work_dir = _env("WORK_DIR")
final_vc_dir = _env("DIR_FINAL_VC")

manifest_rows = []
for voice_id in sorted(os.listdir(final_vc_dir)):
    voice_dir = os.path.join(final_vc_dir, voice_id)
    if not os.path.isdir(voice_dir):
        continue

    ljspeech_rows = []
    for fname in sorted(f for f in os.listdir(voice_dir) if f.endswith(".wav")):
        if fname not in transcripts:
            continue
        text = transcripts[fname]
        audio_path = os.path.join(voice_dir, fname)
        info = sf.info(audio_path)
        stem = os.path.splitext(fname)[0]
        manifest_rows.append({
            "id": f"{voice_id}-{stem}",
            "text": text,
            "lang": lang,
            "audio": os.path.relpath(audio_path, work_dir),
            "duration_s": round(info.duration, 3),
            "sample_rate": info.samplerate,
            "source": {"kind": "corpus", "name": corpus_name},
            "speaker": voice_id,
        })
        ljspeech_rows.append(f"{stem}|{text}")

    with open(os.path.join(voice_dir, "metadata.csv"), "w", encoding="utf-8") as f:
        f.write("\n".join(ljspeech_rows))

assert manifest_rows, "manifest.jsonl would have no rows"

manifest_path = os.path.join(work_dir, "manifest.jsonl")
with open(manifest_path, "w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

meta = {
    "notebook": "asr2tts.ipynb",
    "row_count": len(manifest_rows),
    "instruments": {
        "audiosronnx": {"package": "audiosronnx", "version": _pkg_version("audiosronnx"),
                        "engine": _env("DENOISE_ENGINE", "gtcrn")},
        "voiceclonnx": {"package": "voiceclonnx", "version": _pkg_version("voiceclonnx"),
                        "engine": _env("VC_ENGINE", "openvoice")},
    },
}
with open(manifest_path + ".meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Manifest: {len(manifest_rows)} row(s) -> {manifest_path}")


Licensed under Apache-2.0. Funded through the NGI0 Commons Fund, established
by NLnet with financial support from the European Commission's Next
Generation Internet programme (grant agreement 101135429).
